# DeepAcr 复现 (Wandera et al., Molecular Cell 2022)

**原始论文**：*Anti-CRISPR prediction using deep learning reveals an inhibitor of Cas13b nucleases*  
**原始仓库**：[BackofenLab/DeepAcr](https://github.com/BackofenLab/DeepAcr)

**方法**：多模型集成深度学习（严格复现原始架构）

**架构说明**：  
DeepAcr 使用 **18 个模型的集成**，包含 6 种不同的网络架构（每种训练 3 份）：

| 变体 | 架构 | 输入 |
|------|------|------|
| LSTM | Conv1D → BiLSTM → Linear | 仅序列 one-hot (21d) |
| LSTMb | Conv1D → BiLSTM → Linear + biophys branch | 序列 + 12d 生物物理特征 |
| GRU | Conv1D → BiGRU → Linear | 仅序列 one-hot (21d) |
| GRUb | Conv1D → BiGRU → Linear + biophys branch | 序列 + 12d 生物物理特征 |
| Linear | Flatten → 4× FC → Linear | 仅序列 one-hot (21d) |
| Linearb | Flatten → 4× FC + biophys branch → Linear | 序列 + 12d 生物物理特征 |

- **One-hot 编码**：21 种氨基酸，顺序来自 `aminoacids.yaml`：A,R,N,D,C,E,Q,G,H,I,L,K,M,F,P,S,T,W,Y,V,U
- **12 维生物物理特征 (z_data)**：氨基酸数、分子量、等电点、负电/正电残基数、消光系数×2、不稳定指数、GRAVY、二级结构比例(helix/turn/sheet)
- **集成预测**：18 个模型 sigmoid 输出取平均，阈值 0.5
- **超参数**：严格复现原始仓库 Networks/ 目录中各网络的默认超参数

**环境**：`lm-hf`（PyTorch + BioPython）

In [1]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score,
)
from Bio.SeqUtils.ProtParam import ProteinAnalysis

BENCHMARKS_DIR = '/home/nemophila/projects/protein_bert/anticrispr_benchmarks'
RESULTS_DIR    = '/home/nemophila/projects/protein_bert/Comparison/results'
SEED = 22
NUM_FEATURES = 21  # 21 种氨基酸 one-hot
N_COPIES = 3       # 每种架构训练 3 份（匹配原始仓库）

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## 1. 加载数据

In [2]:
train_df = pd.read_csv(f'{BENCHMARKS_DIR}/anticrispr_binary.train.csv').dropna().drop_duplicates().reset_index(drop=True)
test_df  = pd.read_csv(f'{BENCHMARKS_DIR}/anticrispr_binary.test.csv').dropna().drop_duplicates().reset_index(drop=True)

train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# 计算 max_length（匹配原始仓库：使用训练数据的最大序列长度）
all_seqs = pd.concat([train_df, val_df])['seq'].tolist()
MAX_LEN = max(len(s) for s in all_seqs)
print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')
print(f'MAX_LEN (from train+val): {MAX_LEN}')

train: 996, val: 111, test: 286
MAX_LEN (from train+val): 350


## 2. 特征提取

**One-hot 编码**：严格按照原始仓库 `aminoacids.yaml` 的 21 种氨基酸顺序  
**12 维生物物理特征**：严格复现 `database_mem.py` 中的 `analysis()` 函数，计算后按训练集最大值归一化

In [3]:
# ===== One-hot 编码（aminoacids.yaml 顺序）=====
# A:1, R:2, N:3, D:4, C:5, E:6, Q:7, G:8, H:9, I:10,
# L:11, K:12, M:13, F:14, P:15, S:16, T:17, W:18, Y:19, V:20, U:21
AA_ORDER = 'ARNDCEQGHILKMFPSTWYVU'
AA_DICT = {aa: idx for idx, aa in enumerate(AA_ORDER)}  # 0-indexed for one-hot

def onehot_encode_seq(seq, max_len):
    """One-hot 编码，zero-pad/truncate 至 max_len，匹配原始 create_one_hot_vector + zeropad_or_cut"""
    seq = seq[:max_len]
    encoded = []
    for ch in seq:
        vec = [0] * NUM_FEATURES
        if ch in AA_DICT:
            vec[AA_DICT[ch]] = 1
        encoded.append(vec)
    # zero-pad
    for _ in range(max_len - len(encoded)):
        encoded.append([0] * NUM_FEATURES)
    return encoded


# ===== 12 维生物物理特征（复现 database_mem.py::analysis()）=====
def compute_biophys_features(seq):
    """复现原始 analysis() 函数的 12 维特征。
    原始代码中 sample_sequence[:-1] 是因为 FASTA 序列末尾带 '*' 终止符。
    当前数据集序列不含终止符，因此不做截断以保留完整序列信息。
    """
    clean_seq = seq.replace('U', 'S')
    # 移除非标准氨基酸
    valid_aa = set('ACDEFGHIKLMNPQRSTVWY')
    clean_seq = ''.join(c for c in clean_seq if c in valid_aa)
    if len(clean_seq) == 0:
        return [0.0] * 12

    analyse = ProteinAnalysis(clean_seq)
    aa_dict = analyse.count_amino_acids()

    num_amino_acids = float(sum(aa_dict.values()))
    mw = analyse.molecular_weight()
    pI = analyse.isoelectric_point()
    neg_charged = float(aa_dict.get('D', 0) + aa_dict.get('E', 0))
    pos_charged = float(aa_dict.get('K', 0) + aa_dict.get('R', 0))
    ext1 = float(aa_dict.get('Y', 0) * 1490 + aa_dict.get('W', 0) * 5500)
    ext2 = float(aa_dict.get('Y', 0) * 1490 + aa_dict.get('W', 0) * 5500 + aa_dict.get('C', 0) * 125)
    instab = analyse.instability_index()
    gravy = analyse.gravy()
    helix, turn, sheet = analyse.secondary_structure_fraction()

    return [num_amino_acids, mw, pI, neg_charged, pos_charged,
            ext1, ext2, instab, gravy, helix, turn, sheet]


def extract_all_features(df, max_len):
    """提取 one-hot 序列特征和 12 维生物物理特征"""
    X_onehot = []
    Z_biophys = []
    for seq in df['seq']:
        X_onehot.append(onehot_encode_seq(seq, max_len))
        Z_biophys.append(compute_biophys_features(seq))
    X_onehot = np.array(X_onehot, dtype=np.float32)   # (N, max_len, 21)
    Z_biophys = np.array(Z_biophys, dtype=np.float32)  # (N, 12)
    return X_onehot, Z_biophys


print('Extracting features for train...')
X_train, Z_train = extract_all_features(train_df, MAX_LEN)
print('Extracting features for val...')
X_val, Z_val = extract_all_features(val_df, MAX_LEN)
print('Extracting features for test...')
X_test, Z_test = extract_all_features(test_df, MAX_LEN)

# z_data 归一化：除以训练集最大值（匹配原始 adjust_zdata）
z_max = np.max(Z_train, axis=0)
z_max[z_max == 0] = 1.0  # 防止除零
Z_train_norm = Z_train / z_max
Z_val_norm   = Z_val / z_max
Z_test_norm  = Z_test / z_max

y_train = train_df['label'].to_numpy(dtype=np.float32)
y_val   = val_df['label'].to_numpy(dtype=np.float32)
y_test  = test_df['label'].to_numpy(dtype=np.float32)

print(f'X_train: {X_train.shape}, Z_train: {Z_train_norm.shape}')
print(f'X_val: {X_val.shape}, Z_val: {Z_val_norm.shape}')
print(f'X_test: {X_test.shape}, Z_test: {Z_test_norm.shape}')

Extracting features for train...
Extracting features for val...
Extracting features for test...
X_train: (996, 350, 21), Z_train: (996, 12)
X_val: (111, 350, 21), Z_val: (111, 12)
X_test: (286, 350, 21), Z_test: (286, 12)


## 3. Dataset 定义

In [4]:
class DeepAcrDataset(Dataset):
    """同时提供 one-hot 序列 (x) 和归一化生物物理特征 (z)"""
    def __init__(self, X_onehot, Z_biophys, labels):
        self.x = torch.tensor(X_onehot, dtype=torch.float32)   # (N, L, 21)
        self.z = torch.tensor(Z_biophys, dtype=torch.float32)   # (N, 12)
        self.y = torch.tensor(labels, dtype=torch.float32)       # (N,)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.z[idx], self.y[idx]


BATCH_SIZE = 30  # 原始仓库默认 batch_size=30

train_ds = DeepAcrDataset(X_train, Z_train_norm, y_train)
val_ds   = DeepAcrDataset(X_val, Z_val_norm, y_val)
test_ds  = DeepAcrDataset(X_test, Z_test_norm, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print('Loaders ready.')

Loaders ready.


## 4. 模型架构定义

严格复现原始仓库 `Networks/` 目录中 6 种模型的结构和默认超参数：
- **LSTM / LSTMb**：Conv1D → BiLSTM（2层）→ Linear
- **GRU / GRUb**：Conv1D → BiGRU（2层）→ Linear
- **Linear / Linearb**：Flatten → 4× FC → Linear

后缀 `b` 的变体额外拼接 12 维生物物理特征分支

In [5]:
# ===== 1. LSTM（无生物物理特征）=====
# 原始：Networks/LSTM.py 默认参数 ks=16, st=9, co=9, dropout=0.5, lo=14
class LSTMa(nn.Module):
    def __init__(self, nf, ml, ks=16, st=9, co=9, dropout=0.5, lo=14):
        super().__init__()
        self.nf = nf
        self.lo = lo
        self.dropout = dropout
        self.convolution = nn.Conv1d(nf, co, kernel_size=ks, stride=st)
        conv_out = int(((ml - ks) / st) + 1)
        self.lstm1 = nn.LSTM(co, lo, bidirectional=True, num_layers=2, dropout=dropout)
        in_val = lo * 2 * conv_out
        self.linear = nn.Linear(in_val, 1)

    def forward(self, x, z=None):
        # x: (B, L, nf)
        B = x.size(0)
        x = x.permute(0, 2, 1)                     # (B, nf, L)
        x = F.relu(self.convolution(x))             # (B, co, conv_out)
        x = x.permute(2, 0, 1)                      # (conv_out, B, co)
        hidden = (torch.zeros(4, B, self.lo, device=x.device),
                  torch.zeros(4, B, self.lo, device=x.device))
        x, _ = self.lstm1(x, hidden)                # (conv_out, B, lo*2)
        x = x.permute(1, 0, 2).contiguous().view(B, -1)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.linear(x)                        # (B, 1)


# ===== 2. LSTMb（带生物物理特征）=====
# 原始：Networks/LSTMb.py 默认参数 ks=15, st=10, co=20, dropout=0.5, lo=10, n1=100, n2=41
class LSTMb(nn.Module):
    def __init__(self, nf, ml, ks=15, st=10, co=20, dropout=0.5, lo=10, n1=100, n2=41):
        super().__init__()
        self.nf = nf
        self.lo = lo
        self.convolution = nn.Conv1d(nf, co, kernel_size=ks, stride=st)
        conv_out = int(((ml - ks) / st) + 1)
        self.lstm1 = nn.LSTM(co, lo, bidirectional=True, dropout=dropout, num_layers=2)
        in_val = lo * 2 * conv_out
        self.lin1_add = nn.Linear(12, n1)
        self.lin2_add = nn.Linear(n1, n2)
        self.linear = nn.Linear(in_val + n2, 1)

    def forward(self, x, z):
        B = x.size(0)
        x = x.permute(0, 2, 1)
        x = F.relu(self.convolution(x))
        x = x.permute(2, 0, 1)
        hidden = (torch.zeros(4, B, self.lo, device=x.device),
                  torch.zeros(4, B, self.lo, device=x.device))
        x, _ = self.lstm1(x, hidden)
        x = x.permute(1, 0, 2).contiguous().view(B, -1)
        z = F.relu(self.lin1_add(z))
        z = F.relu(self.lin2_add(z))
        x = torch.cat([x, z], dim=1)
        return self.linear(x)


# ===== 3. GRU（无生物物理特征）=====
# 原始：Networks/GRU.py 默认参数 co=50, ks=25, st=20, go=20, do=0.01
class GRUa(nn.Module):
    def __init__(self, nf, ml, co=50, ks=25, st=20, go=20, do=0.009999999776482582):
        super().__init__()
        self.go = go
        self.do = do
        self.convolution = nn.Conv1d(nf, co, kernel_size=ks, stride=st)
        conv_out = int(((ml - ks) / st) + 1)
        self.gru = nn.GRU(co, go, bidirectional=True, num_layers=2, dropout=do)
        lin_in = go * 2 * conv_out
        self.linear = nn.Linear(lin_in, 1)

    def forward(self, x, z=None):
        B = x.size(0)
        x = x.permute(0, 2, 1)
        x = F.relu(self.convolution(x))
        x = x.permute(2, 0, 1)
        hidden = torch.zeros(4, B, self.go, device=x.device)
        x, _ = self.gru(x, hidden)
        x = x.permute(1, 0, 2).contiguous().view(B, -1)
        x = F.dropout(x, p=self.do, training=self.training)
        return self.linear(x)


# ===== 4. GRUb（带生物物理特征）=====
# 原始：Networks/GRUb.py 默认参数 co=50, ks=17, st=4, go=16, hn=100, hn2=83, dropout=0.2104
class GRUb(nn.Module):
    def __init__(self, nf, ml, co=50, ks=17, st=4, go=16, hn=100, hn2=83, dropout=0.2104):
        super().__init__()
        self.go = go
        self.dropout = dropout
        self.convolution = nn.Conv1d(nf, co, kernel_size=ks, stride=st)
        conv_out = int(((ml - ks) / st) + 1)
        self.gru = nn.GRU(co, go, bidirectional=True, dropout=dropout, num_layers=2)
        lin_in = go * 2 * conv_out
        self.lin1 = nn.Linear(12, hn)
        self.lin2 = nn.Linear(hn, hn2)
        self.linear = nn.Linear(lin_in + hn2, 1)

    def forward(self, x, z):
        B = x.size(0)
        z = F.relu(self.lin1(z))
        z = F.dropout(z, p=self.dropout, training=self.training)
        z = F.relu(self.lin2(z))
        x = x.permute(0, 2, 1)
        x = F.relu(self.convolution(x))
        x = x.permute(2, 0, 1)
        hidden = torch.zeros(4, B, self.go, device=x.device)
        x, _ = self.gru(x, hidden)
        x = x.permute(1, 0, 2).contiguous().view(B, -1)
        x = torch.cat([x, z], dim=1)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.linear(x)


# ===== 5. Linear（无生物物理特征）=====
# 原始：Networks/Linear.py 默认参数 out1=548, out2=36, out3=500, out4=115, dropout=0.05226
class LinearA(nn.Module):
    def __init__(self, nf, ml, out1=548, out2=36, out3=500, out4=115, dropout=0.05226011067821744):
        super().__init__()
        self.dropout = dropout
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(ml * nf, out1)
        self.linear2 = nn.Linear(out1, out2)
        self.linear3 = nn.Linear(out2, out3)
        self.linear4 = nn.Linear(out3, out4)
        self.linear5 = nn.Linear(out4, 1)

    def forward(self, x, z=None):
        x = self.flatten(x)
        x = F.relu(self.linear1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.linear2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.linear3(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.linear4(x))
        return self.linear5(x)


# ===== 6. Linearb（带生物物理特征）=====
# 原始：Networks/Linearb.py 默认参数 out1=197, out2=870, out3=71, out4=61, hidden_nodes=81, dropout=0.33015
class LinearB(nn.Module):
    def __init__(self, nf, ml, out1=197, out2=870, out3=71, out4=61, hidden_nodes=81, dropout=0.33015396090117194):
        super().__init__()
        self.dropout = dropout
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(ml * nf, out1)
        self.linear2 = nn.Linear(out1, out2)
        self.linear3 = nn.Linear(out2, out3)
        self.linear4 = nn.Linear(out3, out4)
        self.lin1_add = nn.Linear(12, hidden_nodes)
        self.lin2_add = nn.Linear(hidden_nodes, hidden_nodes)
        self.lin3_add = nn.Linear(hidden_nodes, hidden_nodes)
        self.lin_cat = nn.Linear(out4 + hidden_nodes, out4)
        self.linear = nn.Linear(out4, 1)

    def forward(self, x, z):
        x = self.flatten(x)
        x = F.relu(self.linear1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.linear2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.linear3(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.linear4(x))
        z = F.relu(self.lin1_add(z))
        z = F.relu(self.lin2_add(z))
        z = F.dropout(z, p=self.dropout, training=self.training)
        z = F.relu(self.lin3_add(z))
        x = torch.cat([x, z], dim=1)
        x = F.relu(self.lin_cat(x))
        return self.linear(x)


# 所有 6 种架构及其是否使用 z_data
MODEL_CONFIGS = [
    ('LSTMa',   LSTMa,   False),
    ('LSTMb',   LSTMb,   True),
    ('GRUa',    GRUa,    False),
    ('GRUb',    GRUb,    True),
    ('LinearA', LinearA, False),
    ('LinearB', LinearB, True),
]

# 验证所有模型可以实例化
for name, cls, uses_z in MODEL_CONFIGS:
    m = cls(NUM_FEATURES, MAX_LEN).to(device)
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name}: {n_params:,} params')
    del m

LSTMa: 11,826 params
LSTMb: 17,603 params
GRUa: 43,061 params
GRUb: 41,683 params
LinearA: 4,124,343 params
LinearB: 1,709,762 params


## 5. 训练 18 个模型

每种架构训练 3 份 → 6 × 3 = 18 个模型  
匹配原始仓库的集成策略：`models_with_infos` + `models_without_infos`，每类 3 个 LSTM + 3 个 GRU + 3 个 LINEAR

In [6]:
def train_one_model(model, train_loader, val_loader, train_ds, val_ds,
                    uses_z, epochs=80, lr=1e-3, patience_max=10):
    """训练单个模型，使用 BCEWithLogitsLoss + early stopping"""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_val_loss = float('inf')
    patience = 0
    best_state = None

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0.0
        for xb, zb, yb in train_loader:
            xb, zb, yb = xb.to(device), zb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb, zb if uses_z else None).squeeze(-1)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(yb)
        train_loss /= len(train_ds)

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, zb, yb in val_loader:
                xb, zb, yb = xb.to(device), zb.to(device), yb.to(device)
                logits = model(xb, zb if uses_z else None).squeeze(-1)
                val_loss += criterion(logits, yb).item() * len(yb)
        val_loss /= len(val_ds)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        if patience >= patience_max:
            break

    model.load_state_dict(best_state)
    return model, best_val_loss


# ===== 训练所有 18 个模型 =====
all_models = []  # list of (name, model, uses_z)

for name, cls, uses_z in MODEL_CONFIGS:
    for copy_id in range(N_COPIES):
        seed_i = SEED + copy_id * 100
        torch.manual_seed(seed_i)
        np.random.seed(seed_i)

        model = cls(NUM_FEATURES, MAX_LEN).to(device)
        model, vloss = train_one_model(
            model, train_loader, val_loader, train_ds, val_ds, uses_z
        )
        all_models.append((f'{name}_{copy_id}', model, uses_z))
        print(f'  {name}_{copy_id}  val_loss={vloss:.4f}')

print(f'\nTotal models trained: {len(all_models)}')

  LSTMa_0  val_loss=0.4341
  LSTMa_1  val_loss=0.4257
  LSTMa_2  val_loss=0.4202
  LSTMb_0  val_loss=0.4160
  LSTMb_1  val_loss=0.3929
  LSTMb_2  val_loss=0.4085
  GRUa_0  val_loss=0.4318
  GRUa_1  val_loss=0.4011
  GRUa_2  val_loss=0.4292
  GRUb_0  val_loss=0.4080
  GRUb_1  val_loss=0.3849
  GRUb_2  val_loss=0.3830
  LinearA_0  val_loss=0.4310
  LinearA_1  val_loss=0.4312
  LinearA_2  val_loss=0.4320
  LinearB_0  val_loss=0.4105
  LinearB_1  val_loss=0.4076
  LinearB_2  val_loss=0.4261

Total models trained: 18


## 6. 集成评估

匹配原始仓库 `DeepAcr_masterscript.py`：对所有 18 个模型的 sigmoid 输出取平均

In [7]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        m = ids == b
        if np.any(m):
            ece += (np.sum(m) / n) * abs(float(np.mean(y_true[m])) - float(np.mean(y_prob[m])))
    return float(ece)


def evaluate_binary_full(y_true, y_prob, threshold=0.5):
    y_cls = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_cls).ravel()
    return {
        'AUC':  float(roc_auc_score(y_true, y_prob)),
        'AUPRC': float(average_precision_score(y_true, y_prob)),
        'F1':   float(f1_score(y_true, y_cls)),
        'MCC':  float(matthews_corrcoef(y_true, y_cls)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'ECE':  expected_calibration_error(y_true, y_prob),
        'ACC':  float(accuracy_score(y_true, y_cls)),
        'SN':   float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
        'SP':   float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
        'Threshold': float(threshold),
    }


def predict_ensemble(models_list, loader):
    """集成预测：对所有模型的 sigmoid 输出取平均（匹配原始 DeepAcr_masterscript.py）"""
    all_predictions = []

    for name, model, uses_z in models_list:
        model.eval()
        preds = []
        with torch.no_grad():
            for xb, zb, yb in loader:
                xb, zb = xb.to(device), zb.to(device)
                logits = model(xb, zb if uses_z else None).squeeze(-1)
                preds.append(torch.sigmoid(logits).cpu().numpy())
        all_predictions.append(np.concatenate(preds))

    # 取所有模型的平均（匹配原始 pred_v = np.mean(predictions, axis=0)）
    ensemble_prob = np.mean(all_predictions, axis=0)
    return ensemble_prob


# ===== 验证集找最优阈值 =====
val_prob = predict_ensemble(all_models, val_loader)
y_val_np = val_df['label'].to_numpy(dtype=int)
best_thr, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.9, 0.01):
    f = f1_score(y_val_np, (val_prob >= t).astype(int))
    if f > best_f1:
        best_f1, best_thr = f, t
print(f'Best threshold (val F1={best_f1:.4f}): {best_thr:.2f}')

# ===== 测试集集成评估 =====
y_test_np = test_df['label'].to_numpy(dtype=int)
y_prob = predict_ensemble(all_models, test_loader)

metrics = evaluate_binary_full(y_test_np, y_prob, threshold=best_thr)
for k, v in metrics.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')

# 打印各模型单独预测的 AUC 信息
print('\n--- 各模型单独 AUC ---')
for name, model, uses_z in all_models:
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, zb, yb in test_loader:
            xb, zb = xb.to(device), zb.to(device)
            logits = model(xb, zb if uses_z else None).squeeze(-1)
            preds.append(torch.sigmoid(logits).cpu().numpy())
    p = np.concatenate(preds)
    auc = roc_auc_score(y_test_np, p)
    print(f'  {name}: AUC={auc:.4f}')

Best threshold (val F1=0.5909): 0.27
AUC: 0.8070
AUPRC: 0.4235
F1: 0.3500
MCC: 0.2825
Brier: 0.0801
ECE: 0.1055
ACC: 0.8182
SN: 0.5385
SP: 0.8462
Threshold: 0.2700

--- 各模型单独 AUC ---
  LSTMa_0: AUC=0.7234
  LSTMa_1: AUC=0.7265
  LSTMa_2: AUC=0.7155
  LSTMb_0: AUC=0.8136
  LSTMb_1: AUC=0.8016
  LSTMb_2: AUC=0.7922
  GRUa_0: AUC=0.8084
  GRUa_1: AUC=0.7791
  GRUa_2: AUC=0.8030
  GRUb_0: AUC=0.8059
  GRUb_1: AUC=0.7896
  GRUb_2: AUC=0.8291
  LinearA_0: AUC=0.6654
  LinearA_1: AUC=0.6555
  LinearA_2: AUC=0.6442
  LinearB_0: AUC=0.7099
  LinearB_1: AUC=0.7178
  LinearB_2: AUC=0.7021


In [8]:
result = {'method': 'DeepAcr (18-model ensemble)', 'metrics': metrics}
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(f'{RESULTS_DIR}/deepacr_metrics.json', 'w') as f:
    json.dump(result, f, indent=2)
np.savez(f'{RESULTS_DIR}/deepacr_predictions.npz', y_true=y_test_np, y_prob=y_prob)
print('Results saved.')

Results saved.
